# Agentic AI

Companion notebook to [Agentic AI](https://yattas.com/tutorials/agentic-ai/overview/). The lesson explains the idea by hand: a reasoning loop, a tool call, an observation fed back in. This notebook builds that exact loop from scratch, around a real small open model, so you can watch it actually run.

Runs on Colab's free CPU tier -- the model download takes a minute or two, and each response the model generates takes several seconds on CPU (faster if you switch to a free GPU runtime, but not required).

Use Colab's outline (View -> Table of contents) to jump between sections.

In [ ]:
!pip install -q transformers accelerate

## 1. Loading a Small Open Model

The model here is [Qwen2.5-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct) -- small enough to download and run without a paid API, open-weight, and no Hugging Face account or token required. Nothing about what makes it an "agent" lives in the model download below; that comes from the loop we build in Section 3.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device)
model.eval()
print("Model loaded.")

## 2. A Calculator Tool, Called By Hand

A "tool" is nothing more than an ordinary Python function. Before wiring it up to the model at all, here it is called directly -- there's no magic in the tool itself, only in how it gets invoked later.

In [ ]:
import re

CALC_WRAPPER_RE = re.compile(r"^\s*calculator\((.*)\)\s*$", re.IGNORECASE)
SAFE_BUILTINS = {"len": len, "abs": abs, "round": round, "min": min, "max": max, "sum": sum}

def calculator(expr):
    # Models don't always follow the exact format asked for -- this strips a
    # redundant `calculator(...)` wrapper if the model adds one anyway.
    m = CALC_WRAPPER_RE.match(expr)
    if m:
        expr = m.group(1)
    try:
        return str(eval(expr, {"__builtins__": SAFE_BUILTINS}, {}))
    except Exception as e:
        return f"error: {e}"

calculator("47 * 89 + 12")

## 3. The Reasoning Loop: Thought, Action, Observation

This is the transcript format from the lesson: the model writes a `Thought`, then either an `Action` + `Action Input` (a tool call) or a `Final Answer`. The system prompt below describes the format; the two-turn example after it *shows* the format, which matters more -- a 3-billion-parameter model follows a pattern it's just seen far more reliably than one it's only been told about.

In [ ]:
SYSTEM_PROMPT = """You can use tools to answer questions.

Tools:
- calculator(expression): evaluates a Python arithmetic expression, returns the result.
- lookup(term): looks up a term in a small fact database and returns what it finds. These are facts you were
  never trained on -- check with this tool before answering, rather than guessing. Only decide
  how to respond, including saying you don't have an answer, after checking.

To use a tool, respond with exactly:
Thought: <your reasoning>
Action: <tool name>
Action Input: <input to the tool -- for calculator, a plain expression like "12 * 7", not wrapped in a function call>

You will then be given an Observation with the real result. Continue reasoning from it.

Once you have enough information, respond with exactly:
Thought: <your reasoning>
Final Answer: <your answer>

Only ever take one action per turn."""

# Shown as real prior turns, not described in text -- the model imitates a
# pattern it's seen far more reliably than one it's only been told about.
FEWSHOT = [
    ("user", "What is 12 times 7?"),
    ("assistant", "Thought: I should use the calculator to be sure.\nAction: calculator\nAction Input: 12 * 7"),
    ("user", "Observation: 84"),
    ("assistant", "Thought: I now have the answer.\nFinal Answer: 84"),
    ("user", "What's the team mascot?"),
    ("assistant", "Thought: I don't know this on my own -- it's a specific fact I should look up.\nAction: lookup\nAction Input: mascot"),
    ("user", "Observation: a corgi named Gradient"),
    ("assistant", "Thought: I now have the answer.\nFinal Answer: The team mascot is a corgi named Gradient."),
]

In [ ]:
ACTION_RE = re.compile(r"Action:\s*(\w+)\s*\nAction Input:\s*(.+)")
FINAL_RE = re.compile(r"Final Answer:\s*(.+)")

def generate(messages, max_new_tokens=150):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def run_agent(question, tools, max_steps=4):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for role, content in FEWSHOT:
        messages.append({"role": role, "content": content})
    messages.append({"role": "user", "content": question})

    for step in range(max_steps):
        response = generate(messages)
        print(f"--- step {step} ---\n{response}\n")

        final_match = FINAL_RE.search(response)
        action_match = ACTION_RE.search(response)

        if final_match and (not action_match or final_match.start() < action_match.start()):
            print("FINAL ANSWER:", final_match.group(1).strip())
            return

        if not action_match:
            print("(no action or final answer found -- stopping)")
            return

        tool_name, tool_input = action_match.group(1).strip(), action_match.group(2).strip()
        messages.append({"role": "assistant", "content": response[: action_match.end()]})
        result = tools[tool_name](tool_input) if tool_name in tools else f"error: unknown tool '{tool_name}'"
        messages.append({"role": "user", "content": f"Observation: {result}"})
        print(f"[executed {tool_name}({tool_input!r}) -> {result}]\n")

    print("(max steps reached without a final answer)")

Now run it on the same question from the lesson. Watch the first `Thought` closely -- the model's own mental arithmetic is wrong. The right answer only shows up because the calculator actually ran, and the model read the real result back out of the `Observation` on the next step.

In [ ]:
run_agent("What is 47 times 89, plus 12?", tools={"calculator": calculator})

## 4. A Second Tool the Model Can't Know Without Asking

The calculator above checks work the model could, in principle, do itself -- badly, as just shown, but it has *some* notion of arithmetic. A `lookup` tool is different: it answers questions about facts that were never in the model's training data at all, the same role Roundtable's per-persona lore retrieval plays for its three characters.

In [ ]:
LORE = {
    "mascot": "a corgi named Gradient",
    "team motto": "Ship it and see.",
    "founder's coffee order": "double espresso, no sugar",
}

def lookup(term):
    term = term.strip().lower()
    if term in LORE:
        return LORE[term]
    for key, value in LORE.items():
        if term in key or key in term:
            return value
    return f"no entry found for '{term}'"

lookup("founder's coffee order")

In [ ]:
run_agent("What's the founder's coffee order?", tools={"lookup": lookup})

## 5. Multiple Tool Calls in One Question

Nothing about the loop changes when a question needs more than one tool -- it just goes around more times. Here the model has to look something up, then hand what it finds to the calculator before it can answer.

In [ ]:
run_agent(
    "Look up the team mascot's name, then use the calculator to find out how many letters are in it.",
    tools={"calculator": calculator, "lookup": lookup},
)

That's the whole mechanism: a transcript, a regular expression watching for `Action:`, and a couple of Python functions -- maybe 60 lines total, including the loop itself. Frameworks like LangGraph or AutoGen, covered in the lesson, don't replace this; they formalize it, so the same loop can route between several agents, retry a malformed tool call, or stream partial output, without every project reimplementing it from scratch.